In [1]:
"""
Task A: LGB + XGB + CatBoost 一次性训练 + 三模型平均 + LP 融合提交脚本

流程：
1. 读取 A1.npz，构造和前面强化版一致的图聚合特征：X / A@X / A^2@X / graph_stats；
2. 在官方 train_idx 内做一次 80%/20% 划分；
3. 用同一份 train_idx_80 / test_idx_20 分别训练 LightGBM、XGBoost、CatBoost；
4. 三个 GBDT 模型的概率先做简单平均；
5. LP 验证阶段只用 train_idx_80 标签，避免 test_idx_20 标签泄漏；
6. 在 test_idx_20 上选择 GBDT3_AVG + LP 的最佳融合策略和 per-class threshold；
7. 最终官方 test：GBDT3_AVG 概率 + 使用完整 train_idx 标签得到的 LP 概率，再融合生成 A1.csv。

依赖：
    pip install lightgbm xgboost catboost
"""

import os
import gc
import time
import json
import pickle
import warnings
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, diags, eye as speye, hstack, issparse
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import SelectKBest, chi2, f_classif

try:
    import lightgbm as lgb
except ImportError as e:
    raise ImportError("未安装 lightgbm，请先运行：pip install lightgbm") from e

try:
    import xgboost as xgb
    from xgboost.core import XGBoostError
except ImportError as e:
    raise ImportError("未安装 xgboost，请先运行：pip install xgboost") from e

try:
    from catboost import CatBoostClassifier, Pool, CatBoostError
except ImportError as e:
    raise ImportError("未安装 catboost，请先运行：pip install catboost") from e


DATA_ROOT = "/kaggle/input/datasets/theunforgiven7/afac2026-task1"
DATA_ROOT1 = "/kaggle/working/"
CHECKPOINT_DIR = os.path.join(DATA_ROOT1, "checkpoints")
OUTPUT_NAME = "A1.csv"
SUMMARY_NAME = "gbdt3_graph_8020_lp_ensemble_summary.json"
ENSEMBLE_CKPT_NAME = "cls_gbdt3_graph_8020_lp_ensemble.pkl"


# ─────────────────────────────────────────────
# 数据加载
# ─────────────────────────────────────────────

def load_data():
    npz_path = os.path.join(DATA_ROOT, "A����", "A1.npz")
    data = np.load(npz_path, allow_pickle=True)
    adj = csr_matrix(
        (data["adj_data"], data["adj_indices"], data["adj_indptr"]),
        shape=tuple(data["adj_shape"]),
    )
    features = csr_matrix(
        (data["attr_data"], data["attr_indices"], data["attr_indptr"]),
        shape=tuple(data["attr_shape"]),
    )
    return adj, features, data["labels"], data["train_idx"], data["test_idx"]


# ─────────────────────────────────────────────
# 图特征构造
# ─────────────────────────────────────────────

def symmetrize_adj(adj, add_self=True):
    adj = adj.tocsr().astype(np.float32)
    adj = adj + adj.T
    adj.data = np.ones_like(adj.data, dtype=np.float32)
    adj.eliminate_zeros()
    if add_self:
        adj = adj + speye(adj.shape[0], format="csr", dtype=np.float32)
        adj.data = np.ones_like(adj.data, dtype=np.float32)
        adj.eliminate_zeros()
    return adj.tocsr()


def row_normalize(mat):
    deg = np.asarray(mat.sum(axis=1)).reshape(-1).astype(np.float32)
    deg_inv = np.power(deg, -1.0)
    deg_inv[np.isinf(deg_inv)] = 0.0
    return (diags(deg_inv) @ mat).tocsr(), deg


def build_graph_features(adj, features, feature_config):
    """构造三模型共用的图聚合特征，返回 scipy CSR。"""
    x0 = features.tocsr().astype(np.float32)

    adj_noself = symmetrize_adj(adj, add_self=False)
    deg = np.asarray(adj_noself.sum(axis=1)).reshape(-1).astype(np.float32)

    adj_self = symmetrize_adj(adj, add_self=True)
    trans, deg_self = row_normalize(adj_self)

    blocks = []
    names = []

    if feature_config.get("use_raw_x", True):
        blocks.append(x0)
        names.append(f"X:{x0.shape[1]}")

    x1 = None
    if feature_config.get("use_ax", True):
        x1 = (trans @ x0).tocsr().astype(np.float32)
        blocks.append(x1)
        names.append(f"AX:{x1.shape[1]}")

    if feature_config.get("use_a2x", True):
        if x1 is None:
            x1 = (trans @ x0).tocsr().astype(np.float32)
        x2 = (trans @ x1).tocsr().astype(np.float32)
        blocks.append(x2)
        names.append(f"A2X:{x2.shape[1]}")

    if feature_config.get("use_graph_stats", True):
        attr_sum = np.asarray(x0.sum(axis=1)).reshape(-1).astype(np.float32)
        attr_nnz = np.diff(x0.indptr).astype(np.float32)
        neigh_deg_mean = np.asarray(trans @ deg.reshape(-1, 1)).reshape(-1).astype(np.float32)

        stats = np.column_stack([
            deg,
            np.log1p(deg),
            deg_self,
            np.log1p(deg_self),
            neigh_deg_mean,
            np.log1p(neigh_deg_mean),
            attr_sum,
            np.log1p(np.maximum(attr_sum, 0)),
            attr_nnz,
            np.log1p(attr_nnz),
        ]).astype(np.float32)
        blocks.append(csr_matrix(stats))
        names.append(f"graph_stats:{stats.shape[1]}")

    x_all = hstack(blocks, format="csr", dtype=np.float32)
    x_all.eliminate_zeros()
    density = x_all.nnz / max(1, x_all.shape[0] * x_all.shape[1])
    print(f"特征块: {names}")
    print(f"最终图特征矩阵: shape={x_all.shape}, nnz={x_all.nnz}, density={density:.6f}")
    return x_all


# ─────────────────────────────────────────────
# 通用工具
# ─────────────────────────────────────────────

def dense_size_gb(shape):
    return shape[0] * shape[1] * 4 / (1024 ** 3)


def has_negative_values(x):
    if issparse(x):
        return x.data.size > 0 and np.nanmin(x.data) < 0
    return np.nanmin(x) < 0


def make_sample_weight(labels, trn_idx, num_classes, mode="none", max_weight=5.0):
    y = labels[trn_idx]
    cnt = np.bincount(y, minlength=num_classes).astype(np.float32)

    if mode == "none":
        w = np.ones(num_classes, dtype=np.float32)
    elif mode == "balanced":
        w = len(y) / np.maximum(cnt, 1.0) / num_classes
    elif mode == "sqrt_balanced":
        w = np.sqrt(len(y) / np.maximum(cnt, 1.0) / num_classes)
    else:
        raise ValueError(f"未知 class_weight_mode: {mode}")

    w = np.clip(w, 0.0, max_weight).astype(np.float32)
    return cnt.astype(np.int64), w, w[y]


def align_prob_columns(probs_raw, classes, num_classes):
    probs_raw = np.asarray(probs_raw, dtype=np.float32)
    out = np.zeros((probs_raw.shape[0], num_classes), dtype=np.float32)
    if classes is None:
        classes = np.arange(probs_raw.shape[1])
    for col, cls in enumerate(classes):
        cls = int(cls)
        if 0 <= cls < num_classes and col < probs_raw.shape[1]:
            out[:, cls] = probs_raw[:, col]
    row_sums = out.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    return out / row_sums


def predict_in_batches(predict_fn, x, num_classes, classes=None, batch_size=8192):
    n = x.shape[0]
    out = np.zeros((n, num_classes), dtype=np.float32)
    for s in range(0, n, batch_size):
        e = min(s + batch_size, n)
        probs = predict_fn(x[s:e])
        out[s:e] = align_prob_columns(probs, classes, num_classes)
    return out


def acc_from_probs(probs, labels, idx):
    return float(accuracy_score(labels[idx], probs[idx].argmax(axis=1)))


# ─────────────────────────────────────────────
# LightGBM
# ─────────────────────────────────────────────

def make_lgb_model(config, num_classes):
    return lgb.LGBMClassifier(
        boosting_type="gbdt",
        objective="multiclass",
        num_class=num_classes,
        n_estimators=config["n_estimators"],
        learning_rate=config["learning_rate"],
        num_leaves=config["num_leaves"],
        max_depth=config["max_depth"],
        min_child_samples=config["min_child_samples"],
        subsample=config["subsample"],
        subsample_freq=config["subsample_freq"],
        colsample_bytree=config["colsample_bytree"],
        reg_alpha=config["reg_alpha"],
        reg_lambda=config["reg_lambda"],
        random_state=config["seed"],
        n_jobs=config["n_jobs"],
        verbosity=-1,
    )


def lgb_predict_proba_best(model, x, best_iteration):
    if best_iteration is not None and int(best_iteration) > 0:
        try:
            return model.predict_proba(x, num_iteration=int(best_iteration))
        except TypeError:
            pass
    return model.predict_proba(x)


def train_lgb(x_all, labels, trn, val, num_classes, config):
    print("\n" + "=" * 80)
    print("[1/3] 训练 LightGBM-GRAPH")
    y_trn = labels[trn]
    y_val = labels[val]

    cnt, class_weights, sample_weight = make_sample_weight(
        labels,
        trn,
        num_classes,
        mode=config["class_weight_mode"],
        max_weight=config["max_class_weight"],
    )
    sample_weight_fit = None if config["class_weight_mode"] == "none" else sample_weight
    print(f"LGB 训练标签分布: {cnt.tolist()}")
    print(f"LGB 类别权重 mode={config['class_weight_mode']}: {np.round(class_weights, 3).tolist()}")

    model = make_lgb_model(config, num_classes)
    fit_kwargs = dict(
        X=x_all[trn],
        y=y_trn,
        eval_set=[(x_all[val], y_val)],
        eval_metric=config["eval_metric"],
        callbacks=[
            lgb.early_stopping(stopping_rounds=config["early_stopping_rounds"], verbose=True),
            lgb.log_evaluation(period=config.get("verbose_eval", 50)),
        ],
    )
    if sample_weight_fit is not None:
        fit_kwargs["sample_weight"] = sample_weight_fit

    t0 = time.time()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model.fit(**fit_kwargs)
    train_time = time.time() - t0

    best_iteration = getattr(model, "best_iteration_", None)
    if best_iteration is None or int(best_iteration) <= 0:
        best_iteration = config["n_estimators"]

    classes = getattr(model, "classes_", np.arange(num_classes))
    pred_fn = lambda xb: lgb_predict_proba_best(model, xb, best_iteration)
    all_probs = predict_in_batches(pred_fn, x_all, num_classes, classes=classes, batch_size=config.get("predict_batch_size", 8192))
    train_acc = acc_from_probs(all_probs, labels, trn)
    val_acc = acc_from_probs(all_probs, labels, val)

    print(f"LGB best_iteration={int(best_iteration)} | train_acc={train_acc:.4f} | val_acc={val_acc:.4f} | {train_time:.1f}s")
    ckpt = {
        "model": model,
        "config": config,
        "best_iteration": int(best_iteration),
        "train_acc": train_acc,
        "val_acc": val_acc,
        "feature_shape": tuple(int(v) for v in x_all.shape),
    }
    return ckpt, all_probs


# ─────────────────────────────────────────────
# XGBoost strong
# ─────────────────────────────────────────────

def prepare_selected_or_dense_inputs(x_all, y_all, trn, val, input_cfg, tag):
    max_gb = float(input_cfg.get("dense_max_gb", 12.0))
    selector = None
    selected_k = None
    raw_dense_gb = dense_size_gb(x_all.shape)
    print(f"{tag} dense float32 预计内存: {raw_dense_gb:.2f} GB，阈值: {max_gb:.2f} GB")

    x_work = x_all
    if input_cfg.get("prefer_dense", True) and raw_dense_gb <= max_gb:
        print(f"{tag} 输入模式: dense_full")
        x_work = x_all.toarray().astype(np.float32, copy=False)
        mode = "dense_full"
    elif input_cfg.get("feature_select_if_dense_too_large", True):
        k = int(min(input_cfg.get("select_k", 30000), x_all.shape[1]))
        if k < x_all.shape[1]:
            print(f"{tag} dense_full 超限，先用训练集 SelectKBest: k={k}")
            score_func = f_classif if has_negative_values(x_all) else chi2
            selector = SelectKBest(score_func=score_func, k=k)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                selector.fit(x_all[trn], y_all[trn])
            x_work = selector.transform(x_all).tocsr().astype(np.float32)
            selected_k = int(k)
            selected_dense_gb = dense_size_gb(x_work.shape)
            print(f"{tag} 选择后 shape={x_work.shape}, dense 预计内存={selected_dense_gb:.2f} GB")
            if input_cfg.get("prefer_dense", True) and selected_dense_gb <= max_gb:
                print(f"{tag} 输入模式: dense_selected")
                x_work = x_work.toarray().astype(np.float32, copy=False)
                mode = "dense_selected"
            else:
                print(f"{tag} 输入模式: sparse_selected")
                mode = "sparse_selected"
        else:
            print(f"{tag} select_k >= 原始维度，输入模式: sparse_full")
            mode = "sparse_full"
    else:
        print(f"{tag} 输入模式: sparse_full")
        mode = "sparse_full"

    preprocessor = {
        "mode": mode,
        "selector": selector,
        "selected_k": selected_k,
        "raw_feature_shape": tuple(int(v) for v in x_all.shape),
        "final_feature_shape": tuple(int(v) for v in x_work.shape),
        "dense_max_gb": max_gb,
    }
    return x_work, x_work[trn], x_work[val], preprocessor


def make_xgb_model(config, num_classes, use_constructor_early_stopping=True, device="cpu"):
    params = dict(
        objective="multi:softprob",
        num_class=num_classes,
        n_estimators=config["n_estimators"],
        learning_rate=config["learning_rate"],
        max_depth=config["max_depth"],
        min_child_weight=config["min_child_weight"],
        gamma=config["gamma"],
        subsample=config["subsample"],
        colsample_bytree=config["colsample_bytree"],
        reg_alpha=config["reg_alpha"],
        reg_lambda=config["reg_lambda"],
        tree_method=config["tree_method"],
        max_bin=config["max_bin"],
        random_state=config["seed"],
        n_jobs=config["n_jobs"],
        eval_metric=config["eval_metric"],
        missing=np.nan,
        verbosity=1,
    )
    if device:
        params["device"] = device
    if use_constructor_early_stopping:
        params["early_stopping_rounds"] = config["early_stopping_rounds"]
    return xgb.XGBClassifier(**params)


def fit_xgb_with_fallback(model, x_trn, y_trn, x_val, y_val, sample_weight, config):
    fit_kwargs = dict(
        X=x_trn,
        y=y_trn,
        eval_set=[(x_val, y_val)],
        verbose=config.get("verbose_eval", 25),
    )
    if sample_weight is not None:
        fit_kwargs["sample_weight"] = sample_weight

    try:
        return model.fit(**fit_kwargs), model
    except XGBoostError as e:
        if str(config.get("device", "cpu")).lower() == "cuda":
            print(f"[Warning] CUDA XGBoost 失败，回退 CPU。原始错误: {str(e)[:300]}")
            cpu_model = make_xgb_model(config, int(config["num_classes"]), use_constructor_early_stopping=True, device="cpu")
            return cpu_model.fit(**fit_kwargs), cpu_model
        raise
    except TypeError:
        device = config.get("device", "cpu")
        model2 = make_xgb_model(config, int(config["num_classes"]), use_constructor_early_stopping=False, device=device)
        fit_kwargs["early_stopping_rounds"] = config["early_stopping_rounds"]
        try:
            return model2.fit(**fit_kwargs), model2
        except XGBoostError as e:
            if str(device).lower() == "cuda":
                print(f"[Warning] CUDA XGBoost 失败，回退 CPU。原始错误: {str(e)[:300]}")
                cpu_model = make_xgb_model(config, int(config["num_classes"]), use_constructor_early_stopping=False, device="cpu")
                return cpu_model.fit(**fit_kwargs), cpu_model
            raise


def get_xgb_best_iteration(model):
    for attr in ("best_iteration", "best_iteration_"):
        try:
            value = getattr(model, attr)
            if value is not None:
                return int(value)
        except Exception:
            pass
    try:
        booster = model.get_booster()
        value = getattr(booster, "best_iteration", None)
        if value is not None:
            return int(value)
    except Exception:
        pass
    return None


def xgb_predict_proba_best(model, x, best_iteration):
    if best_iteration is not None and int(best_iteration) >= 0:
        end_iter = int(best_iteration) + 1
        try:
            return model.predict_proba(x, iteration_range=(0, end_iter))
        except TypeError:
            pass
        try:
            booster = model.get_booster()
            best_ntree_limit = getattr(booster, "best_ntree_limit", 0)
            if best_ntree_limit:
                return model.predict_proba(x, ntree_limit=best_ntree_limit)
        except Exception:
            pass
    return model.predict_proba(x)


def train_xgb_model(x_all, labels, trn, val, num_classes, config):
    print("\n" + "=" * 80)
    print("[2/3] 训练 XGBoost-GRAPH-STRONG")
    config["num_classes"] = int(num_classes)
    y_trn = labels[trn]
    y_val = labels[val]

    cnt, class_weights, sample_weight = make_sample_weight(
        labels,
        trn,
        num_classes,
        mode=config["class_weight_mode"],
        max_weight=config["max_class_weight"],
    )
    sample_weight_fit = None if config["class_weight_mode"] == "none" else sample_weight
    print(f"XGB 训练标签分布: {cnt.tolist()}")
    print(f"XGB 类别权重 mode={config['class_weight_mode']}: {np.round(class_weights, 3).tolist()}")

    t_prep = time.time()
    x_work, x_trn, x_val, preprocessor = prepare_selected_or_dense_inputs(
        x_all, labels, trn, val, config["xgb_input"], tag="XGB"
    )
    print(f"XGB 输入准备耗时: {time.time() - t_prep:.1f}s")

    model = make_xgb_model(config, num_classes, use_constructor_early_stopping=True, device=config.get("device", "cpu"))
    t0 = time.time()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        _, model = fit_xgb_with_fallback(model, x_trn, y_trn, x_val, y_val, sample_weight_fit, config)
    train_time = time.time() - t0

    best_iteration = get_xgb_best_iteration(model)
    used_trees = int(config["n_estimators"]) if best_iteration is None else int(best_iteration) + 1

    classes = getattr(model, "classes_", np.arange(num_classes))
    pred_fn = lambda xb: xgb_predict_proba_best(model, xb, best_iteration)
    all_probs = predict_in_batches(pred_fn, x_work, num_classes, classes=classes, batch_size=config.get("predict_batch_size", 8192))
    train_acc = acc_from_probs(all_probs, labels, trn)
    val_acc = acc_from_probs(all_probs, labels, val)

    print(f"XGB best_iteration={best_iteration} | used_trees={used_trees} | train_acc={train_acc:.4f} | val_acc={val_acc:.4f} | {train_time:.1f}s")
    ckpt = {
        "model": model,
        "config": config,
        "xgb_preprocessor": preprocessor,
        "best_iteration": None if best_iteration is None else int(best_iteration),
        "used_trees": int(used_trees),
        "train_acc": train_acc,
        "val_acc": val_acc,
        "feature_shape": tuple(int(v) for v in preprocessor["final_feature_shape"]),
        "raw_feature_shape": tuple(int(v) for v in x_all.shape),
    }
    del x_work, x_trn, x_val
    gc.collect()
    return ckpt, all_probs


# ─────────────────────────────────────────────
# CatBoost strong
# ─────────────────────────────────────────────

def make_cbt_model(config, task_type=None):
    if task_type is None:
        task_type = config.get("task_type", "GPU")

    params = dict(
        loss_function="MultiClass",
        eval_metric=config.get("eval_metric", "Accuracy"),
        iterations=config["iterations"],
        learning_rate=config["learning_rate"],
        depth=config["depth"],
        l2_leaf_reg=config["l2_leaf_reg"],
        random_seed=config["seed"],
        od_type="Iter",
        od_wait=config["early_stopping_rounds"],
        use_best_model=True,
        allow_writing_files=False,
        verbose=config.get("verbose_eval", 50),
        thread_count=config.get("thread_count", -1),
        task_type=task_type,
        bootstrap_type=config.get("bootstrap_type", "Bernoulli"),
        subsample=config.get("subsample", 0.8),
        random_strength=config.get("random_strength", 1.0),
        border_count=config.get("border_count", 128),
    )
    if str(task_type).upper() == "GPU":
        params["devices"] = str(config.get("devices", "0"))
    else:
        params["rsm"] = config.get("rsm", 0.8)
    return CatBoostClassifier(**params)


def fit_cbt_with_fallback(model, x_trn, y_trn, x_val, y_val, sample_weight, config):
    train_pool = Pool(x_trn, label=y_trn, weight=sample_weight)
    val_pool = Pool(x_val, label=y_val)
    try:
        model.fit(train_pool, eval_set=val_pool)
        return model
    except CatBoostError as e:
        if str(config.get("task_type", "GPU")).upper() == "GPU":
            print(f"[Warning] GPU CatBoost 失败，回退 CPU。原始错误: {str(e)[:300]}")
            cpu_model = make_cbt_model(config, task_type="CPU")
            cpu_model.fit(train_pool, eval_set=val_pool)
            return cpu_model
        raise


def get_cbt_best_iteration(model):
    try:
        value = model.get_best_iteration()
        if value is not None:
            return int(value)
    except Exception:
        pass
    try:
        value = getattr(model, "best_iteration_", None)
        if value is not None:
            return int(value)
    except Exception:
        pass
    return None


def train_cbt_model(x_all, labels, trn, val, num_classes, config):
    print("\n" + "=" * 80)
    print("[3/3] 训练 CatBoost-GRAPH-STRONG")
    y_trn = labels[trn]
    y_val = labels[val]

    cnt, class_weights, sample_weight = make_sample_weight(
        labels,
        trn,
        num_classes,
        mode=config["class_weight_mode"],
        max_weight=config["max_class_weight"],
    )
    sample_weight_fit = None if config["class_weight_mode"] == "none" else sample_weight
    print(f"CBT 训练标签分布: {cnt.tolist()}")
    print(f"CBT 类别权重 mode={config['class_weight_mode']}: {np.round(class_weights, 3).tolist()}")

    t_prep = time.time()
    x_work, x_trn, x_val, preprocessor = prepare_selected_or_dense_inputs(
        x_all, labels, trn, val, config["cbt_input"], tag="CBT"
    )
    print(f"CBT 输入准备耗时: {time.time() - t_prep:.1f}s")

    model = make_cbt_model(config, task_type=config.get("task_type", "GPU"))
    t0 = time.time()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model = fit_cbt_with_fallback(model, x_trn, y_trn, x_val, y_val, sample_weight_fit, config)
    train_time = time.time() - t0

    best_iteration = get_cbt_best_iteration(model)
    used_trees = int(config["iterations"]) if best_iteration is None else int(best_iteration) + 1

    classes = getattr(model, "classes_", np.arange(num_classes))
    pred_fn = lambda xb: model.predict_proba(xb)
    all_probs = predict_in_batches(pred_fn, x_work, num_classes, classes=classes, batch_size=config.get("predict_batch_size", 8192))
    train_acc = acc_from_probs(all_probs, labels, trn)
    val_acc = acc_from_probs(all_probs, labels, val)

    print(f"CBT best_iteration={best_iteration} | used_trees={used_trees} | train_acc={train_acc:.4f} | val_acc={val_acc:.4f} | {train_time:.1f}s")
    ckpt = {
        "model": model,
        "config": config,
        "cbt_preprocessor": preprocessor,
        "best_iteration": None if best_iteration is None else int(best_iteration),
        "used_trees": int(used_trees),
        "train_acc": train_acc,
        "val_acc": val_acc,
        "feature_shape": tuple(int(v) for v in preprocessor["final_feature_shape"]),
        "raw_feature_shape": tuple(int(v) for v in x_all.shape),
    }
    del x_work, x_trn, x_val
    gc.collect()
    return ckpt, all_probs


# ─────────────────────────────────────────────
# Label Propagation
# ─────────────────────────────────────────────

def build_lp_transition(adj):
    n_nodes = adj.shape[0]
    adj_sym = adj + adj.T
    adj_sym.data = np.ones_like(adj_sym.data, dtype=np.float32)
    adj_sym = adj_sym + speye(n_nodes, format="csr", dtype=np.float32)
    deg = np.asarray(adj_sym.sum(axis=1)).reshape(-1)
    deg_inv = np.power(deg, -1.0)
    deg_inv[np.isinf(deg_inv)] = 0.0
    return diags(deg_inv) @ adj_sym


def run_lp_with_seed(transition, labels, seed_idx, num_classes, num_iters=30, alpha=0.2, verbose=False):
    n_nodes = transition.shape[0]
    seed_idx = np.asarray(seed_idx, dtype=np.int64)
    seed_labels = labels[seed_idx].astype(np.int64)

    y = np.ones((n_nodes, num_classes), dtype=np.float64) / num_classes
    y[seed_idx] = 0.0
    y[seed_idx, seed_labels] = 1.0
    y_init = y.copy()

    for it in range(num_iters):
        y_new = (1 - alpha) * (transition @ y) + alpha * y_init
        y_new[seed_idx] = 0.0
        y_new[seed_idx, seed_labels] = 1.0
        diff = np.abs(y_new - y).max()
        y = y_new
        if verbose and ((it + 1) % 10 == 0):
            print(f"  LP iter {it + 1}: max_diff={diff:.6f}")
        if diff < 1e-6:
            if verbose:
                print(f"  LP converged at iter {it + 1}")
            break

    row_sums = y.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    return (y / row_sums).astype(np.float32)


# ─────────────────────────────────────────────
# GBDT3 平均 + LP 融合策略
# ─────────────────────────────────────────────

def _deg_weighted(gbdt_probs, lp_probs, deg, median_deg, std_deg):
    weight = 1.0 / (1.0 + np.exp(-(deg - median_deg) / max(std_deg, 1.0)))
    weight = weight.reshape(-1, 1)
    return weight * gbdt_probs + (1 - weight) * lp_probs


def _deg_linear(gbdt_probs, lp_probs, deg, median_deg, std_deg):
    weight = np.clip((deg - median_deg + std_deg) / max(2 * std_deg, 1.0), 0, 1)
    weight = weight.reshape(-1, 1)
    return weight * gbdt_probs + (1 - weight) * lp_probs


def _deg_lp_heavy(gbdt_probs, lp_probs, deg, median_deg, std_deg):
    weight = 1.0 / (1.0 + np.exp(-(deg - median_deg) / max(std_deg, 1.0)))
    weight = weight.reshape(-1, 1)
    gbdt_w = 0.1 + 0.3 * weight
    return gbdt_w * gbdt_probs + (1 - gbdt_w) * lp_probs


def _deg_lp_heavy2(gbdt_probs, lp_probs, deg, median_deg, std_deg):
    weight = 1.0 / (1.0 + np.exp(-(deg - median_deg) / max(std_deg, 1.0)))
    weight = weight.reshape(-1, 1)
    gbdt_w = 0.05 + 0.25 * weight
    return gbdt_w * gbdt_probs + (1 - gbdt_w) * lp_probs


def _deg_lp_heavy3(gbdt_probs, lp_probs, deg, median_deg, std_deg):
    weight = 1.0 / (1.0 + np.exp(-(deg - median_deg) / max(std_deg, 1.0)))
    weight = weight.reshape(-1, 1)
    gbdt_w = 0.15 + 0.35 * weight
    return gbdt_w * gbdt_probs + (1 - gbdt_w) * lp_probs


def select_strategy_and_thresholds(gbdt_oof_probs, lp_oof_probs, labels, val_idx, deg):
    median_deg = np.median(deg)
    std_deg = deg.std()
    print(f"\nDegree stats: median={median_deg:.1f}, std={std_deg:.1f}, min={deg.min():.0f}, max={deg.max():.0f}")

    strategies = {
        "gbdt3_only": lambda g, l, d: g,
        "lp_only": lambda g, l, d: l,
        "average": lambda g, l, d: 0.5 * g + 0.5 * l,
        "gbdt_heavy": lambda g, l, d: 0.7 * g + 0.3 * l,
        "lp_heavy": lambda g, l, d: 0.3 * g + 0.7 * l,
        "gbdt_vheavy": lambda g, l, d: 0.8 * g + 0.2 * l,
        "lp_0208": lambda g, l, d: 0.2 * g + 0.8 * l,
        "lp_0109": lambda g, l, d: 0.1 * g + 0.9 * l,
        "lp_025": lambda g, l, d: 0.25 * g + 0.75 * l,
        "lp_035": lambda g, l, d: 0.35 * g + 0.65 * l,
        "lp_040": lambda g, l, d: 0.4 * g + 0.6 * l,
        "deg_sigmoid": lambda g, l, d: _deg_weighted(g, l, d, median_deg, std_deg),
        "deg_linear": lambda g, l, d: _deg_linear(g, l, d, median_deg, std_deg),
        "deg_sigmoid2": lambda g, l, d: _deg_weighted(g, l, d, median_deg, std_deg * 2),
        "deg_sigmoid3": lambda g, l, d: _deg_weighted(g, l, d, median_deg, std_deg * 0.5),
        "deg_lp_heavy": lambda g, l, d: _deg_lp_heavy(g, l, d, median_deg, std_deg),
        "deg_lp_heavy2": lambda g, l, d: _deg_lp_heavy2(g, l, d, median_deg, std_deg),
        "deg_lp_heavy3": lambda g, l, d: _deg_lp_heavy3(g, l, d, median_deg, std_deg),
    }

    print("\n[Validation] GBDT3_AVG + LP 融合策略搜索:")
    best_strategy = None
    best_val_acc = -1.0
    strategy_scores = {}
    for name, fn in strategies.items():
        probs = fn(gbdt_oof_probs, lp_oof_probs, deg)
        val_pred = probs[val_idx].argmax(axis=1)
        val_acc = float((val_pred == labels[val_idx]).mean())
        strategy_scores[name] = val_acc
        print(f"  {name:15s}: {val_acc:.4f}")
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_strategy = name

    print(f"\nBest strategy before threshold: {best_strategy} ({best_val_acc:.4f})")
    fn = strategies[best_strategy]
    val_probs = fn(gbdt_oof_probs, lp_oof_probs, deg)[val_idx]
    val_labels = labels[val_idx]
    num_classes = val_probs.shape[1]
    thresholds = np.ones(num_classes, dtype=np.float32)

    # 轻量级 per-class boost 搜索，和之前推理脚本保持一致。
    for cls in range(num_classes):
        for boost in [1.0, 1.02, 1.05, 1.08, 1.10]:
            trial = thresholds.copy()
            trial[cls] = boost
            pred = (val_probs * trial).argmax(axis=1)
            acc = float((pred == val_labels).mean())
            if acc > best_val_acc:
                best_val_acc = acc
                thresholds[cls] = boost

    print(f"After threshold opt: {best_val_acc:.4f}")
    print(f"Thresholds: {thresholds.round(3).tolist()}")

    return best_strategy, strategies[best_strategy], thresholds, float(best_val_acc), strategy_scores


# ─────────────────────────────────────────────
# 主流程
# ─────────────────────────────────────────────

def get_config():
    return {
        "model_type": "GBDT3_GRAPH_8020_LP_ENSEMBLE",
        "test_size": 0.2,
        "seed": 42,
        "split_type": "train_test_split_80_20_stratify_if_possible",
        "feature_config": {
            "use_raw_x": True,
            "use_ax": True,
            "use_a2x": True,
            "use_graph_stats": True,
        },
        "lp": {
            "num_iters": 30,
            "alpha": 0.2,
        },
        "lgb": {
            "seed": 42,
            "n_estimators": 5000,
            "learning_rate": 0.03,
            "early_stopping_rounds": 200,
            "num_leaves": 127,
            "max_depth": -1,
            "min_child_samples": 20,
            "subsample": 0.85,
            "subsample_freq": 1,
            "colsample_bytree": 0.8,
            "reg_alpha": 0.0,
            "reg_lambda": 2.0,
            "class_weight_mode": "none",
            "max_class_weight": 5.0,
            "n_jobs": -1,
            "eval_metric": "multi_error",
            "verbose_eval": 50,
            "predict_batch_size": 8192,
        },
        "xgb": {
            "seed": 42,
            "xgb_input": {
                "prefer_dense": True,
                "dense_max_gb": 12.0,
                "feature_select_if_dense_too_large": True,
                "select_k": 30000,
            },
            "n_estimators": 3000,
            "learning_rate": 0.04,
            "early_stopping_rounds": 150,
            "max_depth": 6,
            "min_child_weight": 1.0,
            "gamma": 0.0,
            "subsample": 0.9,
            "colsample_bytree": 0.9,
            "reg_alpha": 0.0,
            "reg_lambda": 2.0,
            "tree_method": "hist",
            "max_bin": 256,
            "device": "cuda",
            "class_weight_mode": "none",
            "max_class_weight": 5.0,
            "n_jobs": -1,
            "eval_metric": "merror",
            "verbose_eval": 25,
            "predict_batch_size": 8192,
        },
        "cbt": {
            "seed": 42,
            "cbt_input": {
                "prefer_dense": True,
                "dense_max_gb": 12.0,
                "feature_select_if_dense_too_large": True,
                "select_k": 30000,
            },
            "iterations": 3000,
            "learning_rate": 0.04,
            "early_stopping_rounds": 150,
            "depth": 6,
            "l2_leaf_reg": 6.0,
            "bootstrap_type": "Bernoulli",
            "subsample": 0.85,
            "random_strength": 1.0,
            "border_count": 128,
            "task_type": "GPU",
            "devices": "0",
            "rsm": 0.8,
            "class_weight_mode": "none",
            "max_class_weight": 5.0,
            "thread_count": -1,
            "eval_metric": "Accuracy",
            "verbose_eval": 50,
            "predict_batch_size": 8192,
        },
    }


def train_and_predict():
    print("=" * 80)
    print("  LGB + XGB + CatBoost 平均融合，再和 LP 融合")
    print("=" * 80)

    config = get_config()
    print(f"配置: {json.dumps(config, indent=2, ensure_ascii=False)}")

    print("\n加载数据...")
    adj, features, labels, train_idx, official_test_idx = load_data()
    labels = labels.astype(np.int64)
    train_idx = np.asarray(train_idx, dtype=np.int64)
    official_test_idx = np.asarray(official_test_idx, dtype=np.int64)

    num_nodes = len(labels)
    num_classes = int(labels[train_idx].max()) + 1
    print(f"总节点数: {num_nodes}")
    print(f"原始特征维度: {features.shape[1]}")
    print(f"类别数: {num_classes}")
    print(f"训练标注数: {len(train_idx)}, 官方测试数: {len(official_test_idx)}")
    print(f"全训练集标签分布: {np.bincount(labels[train_idx], minlength=num_classes).tolist()}")
    print(f"图邻接矩阵: shape={adj.shape}, nnz={adj.nnz}")

    np.random.seed(config["seed"])
    stratify = labels[train_idx]
    if np.min(np.bincount(stratify, minlength=num_classes)) < 2:
        print("[Warning] 至少一个类别样本数 < 2，无法 stratify，回退普通随机 80/20。")
        stratify = None

    trn, val = train_test_split(
        train_idx,
        test_size=config["test_size"],
        shuffle=True,
        random_state=config["seed"],
        stratify=stratify,
    )
    trn = np.asarray(trn, dtype=np.int64)
    val = np.asarray(val, dtype=np.int64)

    print("\n" + "=" * 80)
    print("80%/20% 划分结果")
    print(f"训练: {len(trn)}, 验证: {len(val)}")
    print(f"训练标签分布: {np.bincount(labels[trn], minlength=num_classes).tolist()}")
    print(f"验证标签分布: {np.bincount(labels[val], minlength=num_classes).tolist()}")

    print("\n构造三模型共用图聚合特征...")
    t_feat = time.time()
    x_all = build_graph_features(adj, features, config["feature_config"])
    print(f"特征构造耗时: {time.time() - t_feat:.1f}s")

    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

    # 1. 三个 GBDT 模型分别训练。
    lgb_ckpt, lgb_all_probs = train_lgb(x_all, labels, trn, val, num_classes, config["lgb"])
    with open(os.path.join(CHECKPOINT_DIR, "cls_gbdt3_lgb_part.pkl"), "wb") as f:
        pickle.dump({**lgb_ckpt, "train_idx_80": trn, "test_idx_20": val, "num_classes": num_classes}, f)

    xgb_ckpt, xgb_all_probs = train_xgb_model(x_all, labels, trn, val, num_classes, config["xgb"])
    with open(os.path.join(CHECKPOINT_DIR, "cls_gbdt3_xgb_part.pkl"), "wb") as f:
        pickle.dump({**xgb_ckpt, "train_idx_80": trn, "test_idx_20": val, "num_classes": num_classes}, f)

    cbt_ckpt, cbt_all_probs = train_cbt_model(x_all, labels, trn, val, num_classes, config["cbt"])
    with open(os.path.join(CHECKPOINT_DIR, "cls_gbdt3_cbt_part.pkl"), "wb") as f:
        pickle.dump({**cbt_ckpt, "train_idx_80": trn, "test_idx_20": val, "num_classes": num_classes}, f)

    # 2. 三模型简单平均。用户指定：先平均，再和 LP 融合。
    print("\n" + "=" * 80)
    print("三模型概率简单平均")
    gbdt3_all_probs = lgb_all_probs * 0.3 + xgb_all_probs * 0.45 + cbt_all_probs * 0.25
    model_scores = {
        "lgb_train_acc": acc_from_probs(lgb_all_probs, labels, trn),
        "lgb_val_acc": acc_from_probs(lgb_all_probs, labels, val),
        "xgb_train_acc": acc_from_probs(xgb_all_probs, labels, trn),
        "xgb_val_acc": acc_from_probs(xgb_all_probs, labels, val),
        "cbt_train_acc": acc_from_probs(cbt_all_probs, labels, trn),
        "cbt_val_acc": acc_from_probs(cbt_all_probs, labels, val),
        "gbdt3_avg_train_acc": acc_from_probs(gbdt3_all_probs, labels, trn),
        "gbdt3_avg_val_acc": acc_from_probs(gbdt3_all_probs, labels, val),
    }
    for k, v in model_scores.items():
        print(f"{k}: {v:.4f}")

    # 3. LP：验证只用 80% train seed，最终官方 test 用完整 train_idx seed。
    print("\n" + "=" * 80)
    print("Label Propagation")
    transition = build_lp_transition(adj)
    t_lp = time.time()
    print("[LP validation] 只用 train_idx_80 标签作为 seed，避免验证泄漏...")
    lp_oof_probs = run_lp_with_seed(
        transition,
        labels,
        trn,
        num_classes=num_classes,
        num_iters=config["lp"]["num_iters"],
        alpha=config["lp"]["alpha"],
        verbose=False,
    )
    lp_val_acc = acc_from_probs(lp_oof_probs, labels, val)
    print(f"LP 20% val_acc={lp_val_acc:.4f}")

    print("[LP final] 使用完整 train_idx 标签作为 seed，生成官方 test LP 概率...")
    lp_all_probs = run_lp_with_seed(
        transition,
        labels,
        train_idx,
        num_classes=num_classes,
        num_iters=config["lp"]["num_iters"],
        alpha=config["lp"]["alpha"],
        verbose=True,
    )
    print(f"LP 总耗时: {time.time() - t_lp:.1f}s")

    # 4. 度数与融合策略选择。
    adj_sym = adj + adj.T
    adj_sym.data = np.ones_like(adj_sym.data)
    deg = np.asarray(adj_sym.sum(axis=1)).reshape(-1).astype(np.float32)

    best_strategy, best_fn, thresholds, best_val_acc, strategy_scores = select_strategy_and_thresholds(
        gbdt_oof_probs=gbdt3_all_probs,
        lp_oof_probs=lp_oof_probs,
        labels=labels,
        val_idx=val,
        deg=deg,
    )

    # 5. 最终官方 test 预测。
    final_probs = best_fn(gbdt3_all_probs, lp_all_probs, deg)
    final_probs_adj = final_probs * thresholds.reshape(1, -1)
    official_test_pred = final_probs_adj[official_test_idx].argmax(axis=1).astype(np.int64)

    sample_path = os.path.join(DATA_ROOT, "A����", "sample_submission.csv")
    sample = pd.read_csv(sample_path)
    sample["label"] = official_test_pred
    out_path = os.path.join(DATA_ROOT1, OUTPUT_NAME)
    sample.to_csv(out_path, index=False)

    label_dist = np.bincount(official_test_pred, minlength=num_classes).tolist()
    print("\n" + "=" * 80)
    print("最终结果")
    print(f"Saved: {out_path}")
    print(f"Rows: {len(sample)}")
    print(f"Test label dist: {label_dist}")
    print(f"GBDT3_AVG val_acc: {model_scores['gbdt3_avg_val_acc']:.4f}")
    print(f"LP val_acc: {lp_val_acc:.4f}")
    print(f"Best ensemble: {best_strategy} | val_acc={best_val_acc:.4f}")

    # 6. 保存 ensemble checkpoint 与 summary。
    ensemble_ckpt = {
        "config": config,
        "train_idx_80": trn,
        "test_idx_20": val,
        "official_train_idx": train_idx,
        "official_test_idx": official_test_idx,
        "num_classes": int(num_classes),
        "feature_shape": tuple(int(v) for v in x_all.shape),
        "lgb": lgb_ckpt,
        "xgb": xgb_ckpt,
        "cbt": cbt_ckpt,
        "model_scores": model_scores,
        "lp_val_acc": float(lp_val_acc),
        "best_strategy": best_strategy,
        "best_val_acc": float(best_val_acc),
        "thresholds": thresholds.astype(np.float32),
        "strategy_scores": strategy_scores,
        "test_label_dist": label_dist,
        "output": out_path,
    }
    ensemble_path = os.path.join(CHECKPOINT_DIR, ENSEMBLE_CKPT_NAME)
    with open(ensemble_path, "wb") as f:
        pickle.dump(ensemble_ckpt, f)

    # 单独保存 booster/cbm，方便排查。
    try:
        lgb_ckpt["model"].booster_.save_model(os.path.join(CHECKPOINT_DIR, "cls_gbdt3_lgb_part.txt"), num_iteration=int(lgb_ckpt["best_iteration"]))
    except Exception as e:
        print(f"[Warning] 保存 LGB txt 失败: {e}")
    try:
        xgb_ckpt["model"].save_model(os.path.join(CHECKPOINT_DIR, "cls_gbdt3_xgb_part.json"))
    except Exception as e:
        print(f"[Warning] 保存 XGB json 失败: {e}")
    try:
        cbt_ckpt["model"].save_model(os.path.join(CHECKPOINT_DIR, "cls_gbdt3_cbt_part.cbm"))
    except Exception as e:
        print(f"[Warning] 保存 CBT cbm 失败: {e}")

    summary = {
        "config": config,
        "num_train": int(len(trn)),
        "num_val": int(len(val)),
        "train_label_dist": np.bincount(labels[trn], minlength=num_classes).tolist(),
        "val_label_dist": np.bincount(labels[val], minlength=num_classes).tolist(),
        "feature_shape": tuple(int(v) for v in x_all.shape),
        "model_scores": {k: float(v) for k, v in model_scores.items()},
        "lp_val_acc": float(lp_val_acc),
        "best_strategy": best_strategy,
        "best_val_acc": float(best_val_acc),
        "thresholds": thresholds.round(6).tolist(),
        "strategy_scores": {k: float(v) for k, v in strategy_scores.items()},
        "test_label_dist": label_dist,
        "output": out_path,
        "ensemble_checkpoint": ensemble_path,
    }
    summary_path = os.path.join(CHECKPOINT_DIR, SUMMARY_NAME)
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)

    print(f"Ensemble checkpoint saved: {ensemble_path}")
    print(f"Summary saved: {summary_path}")
    return float(best_val_acc)


if __name__ == "__main__":
    train_and_predict()


  LGB + XGB + CatBoost 平均融合，再和 LP 融合
配置: {
  "model_type": "GBDT3_GRAPH_8020_LP_ENSEMBLE",
  "test_size": 0.2,
  "seed": 42,
  "split_type": "train_test_split_80_20_stratify_if_possible",
  "feature_config": {
    "use_raw_x": true,
    "use_ax": true,
    "use_a2x": true,
    "use_graph_stats": true
  },
  "lp": {
    "num_iters": 30,
    "alpha": 0.2
  },
  "lgb": {
    "seed": 42,
    "n_estimators": 5000,
    "learning_rate": 0.03,
    "early_stopping_rounds": 200,
    "num_leaves": 127,
    "max_depth": -1,
    "min_child_samples": 20,
    "subsample": 0.85,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.0,
    "reg_lambda": 2.0,
    "class_weight_mode": "none",
    "max_class_weight": 5.0,
    "n_jobs": -1,
    "eval_metric": "multi_error",
    "verbose_eval": 50,
    "predict_batch_size": 8192
  },
  "xgb": {
    "seed": 42,
    "xgb_input": {
      "prefer_dense": true,
      "dense_max_gb": 12.0,
      "feature_select_if_dense_too_large": true,
      

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LGB best_iteration=400 | train_acc=1.0000 | val_acc=0.6697 | 807.9s

[2/3] 训练 XGBoost-GRAPH-STRONG
XGB 训练标签分布: [279, 1370, 905, 346, 3301, 198, 311, 524, 1379, 187]
XGB 类别权重 mode=none: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
XGB dense float32 预计内存: 0.12 GB，阈值: 12.00 GB
XGB 输入模式: dense_full
XGB 输入准备耗时: 0.1s
[0]	validation_0-merror:0.62472
[25]	validation_0-merror:0.53294
[50]	validation_0-merror:0.44889
[75]	validation_0-merror:0.41118
[100]	validation_0-merror:0.39346
[125]	validation_0-merror:0.38255
[150]	validation_0-merror:0.37483
[175]	validation_0-merror:0.36801
[200]	validation_0-merror:0.36483
[225]	validation_0-merror:0.36256
[250]	validation_0-merror:0.36029
[275]	validation_0-merror:0.35756
[300]	validation_0-merror:0.35575
[325]	validation_0-merror:0.35348
[350]	validation_0-merror:0.34939
[375]	validation_0-merror:0.34848
[400]	validation_0-merror:0.34893
[425]	validation_0-merror:0.34484
[450]	validation_0-merror:0.34212
[475]	validation_0-merror:0.34212
[500]	

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [01:46:45] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


XGB best_iteration=1486 | used_trees=1487 | train_acc=1.0000 | val_acc=0.6897 | 512.8s

[3/3] 训练 CatBoost-GRAPH-STRONG
CBT 训练标签分布: [279, 1370, 905, 346, 3301, 198, 311, 524, 1379, 187]
CBT 类别权重 mode=none: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
CBT dense float32 预计内存: 0.12 GB，阈值: 12.00 GB
CBT 输入模式: dense_full
CBT 输入准备耗时: 0.1s
0:	learn: 0.4264773	test: 0.4129941	best: 0.4129941 (0)	total: 326ms	remaining: 16m 16s
50:	learn: 0.5378409	test: 0.5029532	best: 0.5029532 (50)	total: 6.21s	remaining: 5m 58s
100:	learn: 0.5938636	test: 0.5361199	best: 0.5383916 (97)	total: 12.1s	remaining: 5m 48s
150:	learn: 0.6418182	test: 0.5638346	best: 0.5647433 (146)	total: 18.1s	remaining: 5m 41s
200:	learn: 0.6832955	test: 0.5765561	best: 0.5783735 (199)	total: 24s	remaining: 5m 34s
250:	learn: 0.7219318	test: 0.5874602	best: 0.5879146 (237)	total: 29.8s	remaining: 5m 26s
300:	learn: 0.7545455	test: 0.5988187	best: 0.5988187 (299)	total: 35.4s	remaining: 5m 17s
350:	learn: 0.7838636	test: 0.60